# RAG versus non-RAG result analysis

This notebook validates and compares the GPT-5.6 Terra prompt variants under RAG and non-RAG conditions. It joins predictions by `question_id`, uses matched samples for direct comparisons, runs exact McNemar tests, and regenerates the auditable artifacts in `rag-terra/result-analysis/output/`.

Run all cells from top to bottom after any prediction CSV changes. The positive class is `true`: the question contains a false or materially misleading medical assumption.

## 1. Setup

The notebook imports the tested standard-library analysis module from this folder. No pandas, matplotlib, or seaborn installation is required.

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
from pathlib import Path

from IPython.display import Markdown, SVG, display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_PATH = PROJECT_ROOT / "rag-terra" / "result-analysis" / "analyze.py"
OUTPUT_DIR = PROJECT_ROOT / "rag-terra" / "result-analysis" / "output"

spec = importlib.util.spec_from_file_location("result_analysis", MODULE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not import {MODULE_PATH}")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Validate inputs and regenerate outputs

This is the authoritative execution step. It fails early on missing files, invalid headers or Boolean values, unknown IDs, or duplicate IDs.

In [ ]:
analysis.main(["--output", str(OUTPUT_DIR)])

## 3. Load generated tables

Small display helpers keep the notebook dependency-free while the full-resolution CSV files remain available for downstream analysis.

In [ ]:
def read_csv(name: str) -> list[dict[str, str]]:
    with (OUTPUT_DIR / name).open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def markdown_table(
    rows: list[dict[str, object]],
    columns: list[tuple[str, str]],
    *,
    limit: int | None = None,
) -> str:
    shown = rows if limit is None else rows[:limit]

    def clean(value: object) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(label for _, label in columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    lines.extend(
        "| " + " | ".join(clean(row.get(key, "")) for key, _ in columns) + " |"
        for row in shown
    )
    return "\n".join(lines)


metrics = read_csv("metrics.csv")
paired = read_csv("paired_comparison.csv")
subgroups = read_csv("subgroup_metrics.csv")
disagreements = read_csv("disagreements.csv")
errors = read_csv("errors.csv")
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text(encoding="utf-8"))

print(f"Dataset records: {manifest['dataset']['records']:,}")
print(f"Six-way matched records: {manifest['six_way_common_records']:,}")
print(f"Subgroup rows: {len(subgroups):,}; disagreements: {len(disagreements):,}; errors: {len(errors):,}")

## 4. Overall matched metrics

These rows use each prompt's RAG/non-RAG question-ID intersection. Accuracy intervals are 95% Wilson score intervals.

In [ ]:
matched_metrics = [row for row in metrics if row["scope"] == "pairwise_matched"]
metric_columns = [
    ("condition", "Condition"),
    ("prompt", "Prompt"),
    ("n", "n"),
    ("accuracy", "Accuracy"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("specificity", "Specificity"),
    ("f1", "F1"),
    ("balanced_accuracy", "Balanced accuracy"),
]
display(Markdown(markdown_table(matched_metrics, metric_columns)))

In [ ]:
display(SVG(filename=str(OUTPUT_DIR / "plots" / "overall_metrics.svg")))
display(SVG(filename=str(OUTPUT_DIR / "plots" / "confusion_matrices.svg")))

## 5. Paired retrieval effect

The accuracy delta is `RAG - non-RAG`. The two-sided exact McNemar test uses only discordant pairs. Its three p-values are exploratory and unadjusted.

In [ ]:
paired_columns = [
    ("prompt", "Prompt"),
    ("n", "Matched n"),
    ("rag_accuracy", "RAG accuracy"),
    ("non_rag_accuracy", "Non-RAG accuracy"),
    ("accuracy_delta", "Accuracy delta"),
    ("rag_only_correct", "RAG only correct"),
    ("non_rag_only_correct", "Non-RAG only correct"),
    ("mcnemar_exact_p", "Exact McNemar p"),
]
display(Markdown(markdown_table(paired, paired_columns)))

In [ ]:
display(SVG(filename=str(OUTPUT_DIR / "plots" / "accuracy_delta.svg")))
display(SVG(filename=str(OUTPUT_DIR / "plots" / "paired_outcomes.svg")))

## 6. Explore subgroup metrics

Choose `split` or `cancer`. The default minimum size suppresses very small cancer groups from this notebook view; all groups remain in `subgroup_metrics.csv`.

In [ ]:
GROUP_TYPE = "split"  # Change to "cancer" for cancer-type rows.
MIN_GROUP_SIZE = 1 if GROUP_TYPE == "split" else 5

selected_subgroups = [
    row
    for row in subgroups
    if row["group_type"] == GROUP_TYPE and int(row["n"]) >= MIN_GROUP_SIZE
]
subgroup_columns = [
    ("group", GROUP_TYPE.title()),
    ("n", "n"),
    ("condition", "Condition"),
    ("prompt", "Prompt"),
    ("prevalence", "Positive prevalence"),
    ("accuracy", "Accuracy"),
    ("recall", "Recall"),
    ("specificity", "Specificity"),
    ("balanced_accuracy", "Balanced accuracy"),
]
display(Markdown(markdown_table(selected_subgroups, subgroup_columns, limit=60)))
print(f"Showing {min(60, len(selected_subgroups)):,} of {len(selected_subgroups):,} selected subgroup rows.")

## 7. Inspect disagreements and errors

Change the prompt and row limit to review cases where retrieval changed the classification. `favored_condition` identifies which prediction matched the reference label.

In [ ]:
PROMPT = "basic"  # basic, oncology_expert, or patient_education
ROW_LIMIT = 20

selected_disagreements = [row for row in disagreements if row["prompt"] == PROMPT]
disagreement_columns = [
    ("question_id", "ID"),
    ("split", "Split"),
    ("cancer", "Cancer"),
    ("correct_answer", "Reference"),
    ("rag_answer", "RAG"),
    ("non_rag_answer", "Non-RAG"),
    ("favored_condition", "Correct condition"),
    ("question", "Question"),
]
display(Markdown(markdown_table(selected_disagreements, disagreement_columns, limit=ROW_LIMIT)))
print(f"Showing {min(ROW_LIMIT, len(selected_disagreements)):,} of {len(selected_disagreements):,} disagreements for {PROMPT}.")

## 8. Generated report and audit files

The report states the main finding and interpretation limits. Use the CSV artifacts for detailed follow-up and `manifest.json` to verify the exact source files.

In [ ]:
report_text = (OUTPUT_DIR / "report.md").read_text(encoding="utf-8")
display(Markdown(report_text.replace("plots/", "output/plots/")))